# L3 step 6: the 4-bit 3B on a laptop with no GPU

**What:** our field model, the 3B (c) (Qwen2.5-VL-3B trained on the 7B's checked answers), in 4 bits, answering the benchmark's questions on a laptop with no GPU. It measures memory, speed and accuracy where the model is meant to run. Goal 4 asks for "a distilled 3B in about 6 GB"; until now it was measured only on a Kaggle T4 (L3: 3.03 GB at peak, 1.72 s per question).

**The laptop:** an ASUS Vivobook (Core i9-13900H, 14 cores, 15.6 GB of memory, Intel Iris Xe graphics only, no NVIDIA GPU), plugged in, with other programs closed. Everything runs on the CPU.

**How:**
- **The program: llama.cpp** (release b12237, 9 Nov 2026), the usual free way to run models like ours on a CPU. bitsandbytes, the 4-bit library of the T4 run, needs an NVIDIA GPU.
- **The model:** `Qwen/Qwen2.5-VL-3B-Instruct` with the adapter `AbrarAlam/disasterm3-qwen25vl3b-l2/c/final` merged in, then converted: the language part to 4 bits (Q4_K_M), the image part to 16 bits. Prepared once by Cell 3.
- **The questions** (`l3-step6-download-inputs.py`, fixed before the run): the 60 that L3 timed (the first 10 of each task) and 300 more (50 per task, seed 0), all from L3's 10,675.
- **Asked exactly as on the T4:** the same prompts, photos scaled by the same rule (at most 602,112 pixels), greedy decoding, at most 32 new tokens, one question at a time. Scored by the same rule (the set of option letters must match).
- **Measured:** peak memory of the llama.cpp server, load time, seconds per question, size on disk, accuracy per task, and each answer against the T4's 4-bit answer to the same question.

**The rule and the expectation, fixed 24 Nov before the run:**

| # | Expectation | Why |
|---|---|---|
| 1 | Peak memory at most 6 GB (likely 3 to 5 GB) | Goal 4's "about 6 GB" |
| 2 | 10 to 60 s per question (the T4 took 1.72 s) | A CPU reads the image tokens far slower than a GPU |
| 3 | No difference beyond chance from the T4's 4-bit answers on the same 360 questions (exact McNemar, p ≥ 0.05) | Both are 4-bit copies of the same model |
| 4 | No garbled ("!!!") or empty answers | The CPU doesn't compute in half precision, where the T4 run overflowed |

The 3B passes as a laptop model if 1, 3 and 4 hold; 2 says whether it is quick enough to use in the field.

**Run:** in Jupyter on the PC, top to bottom. Cell 3 takes about half an hour the first time and is skipped afterwards; Cell 6 takes about 2 to 4 hours and resumes where it stopped. Results go to `L3-results/laptop/`.

In [1]:
# ── Cell 1: Paths, versions and the machine ──
import ast, base64, hashlib, io, json, math, os, platform, re, subprocess, sys, threading, time
from pathlib import Path
import pandas as pd, psutil, requests
from PIL import Image

CT = Path.cwd()                                           # Champion_Task (the notebook runs from its own folder)
WORK = Path("D:/CSE499_laptop_test")                      # outside the repository: models, llama.cpp, photos
INPUTS = WORK / "inputs"
LLAMA = WORK / "llama.cpp-b12237" / "bin"
CONVERT = next((WORK / "llama.cpp-b12237" / "src").glob("*/convert_hf_to_gguf.py"))
BASE, ADAPTER = WORK / "hf" / "base", WORK / "hf" / "adapter" / "c" / "final"
MERGED, GGUF = WORK / "merged-c", WORK / "gguf"
LLM_F16, LLM_Q4, MMPROJ = GGUF / "3b-c-f16.gguf", GGUF / "3b-c-Q4_K_M.gguf", GGUF / "mmproj-3b-c-f16.gguf"
OUT = CT / "L3-results" / "laptop"
OUT.mkdir(parents=True, exist_ok=True)
L3_DIR = CT / "L3-results" / "full" / "L3"
PORT, MAX_NEW_TOKENS, CONTEXT = 8091, 32, 4096
for p in (INPUTS / "questions.json", INPUTS / "benchmark_release.json", LLAMA / "llama-server.exe", BASE, ADAPTER, L3_DIR / "per_question.csv"):
    assert p.exists(), f"missing: {p}"

def cpu_name():
    try:
        return subprocess.run(["powershell", "-NoProfile", "-Command", "(Get-CimInstance Win32_Processor).Name"],
                              capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception:
        return platform.processor()

MACHINE = {"cpu": cpu_name(), "cores": psutil.cpu_count(logical=False), "threads": psutil.cpu_count(),
           "memory_gb": round(psutil.virtual_memory().total / 1e9, 1), "windows": platform.platform(),
           "on_battery": bool(getattr(psutil.sensors_battery(), "power_plugged", True) is False),
           "python": sys.version.split()[0], "llama_cpp": "b12237",
           # pinned when downloaded (24 Nov): the Hugging Face revisions and the llama.cpp zip's SHA-256
           "base_model": "Qwen/Qwen2.5-VL-3B-Instruct@66285546d2b821cf421d4f5eb2576359d3770cd3",
           "adapter": "AbrarAlam/disasterm3-qwen25vl3b-l2@13ce20ba6d0a13dc5a5ddf2ad09e0704897e4d5c (c/final)",
           "llama_cpp_zip_sha256": "1472bc5767155a77811395c7a5504283e3eb7c0aedff113e42d485d3ae00f8c7"}
print(json.dumps(MACHINE, indent=1))
assert not MACHINE["on_battery"], "plug the laptop in"


{
 "cpu": "13th Gen Intel(R) Core(TM) i9-13900H",
 "cores": 14,
 "threads": 20,
 "memory_gb": 16.8,
 "windows": "Windows-11-10.0.26200-SP0",
 "on_battery": false,
 "python": "3.13.5",
 "llama_cpp": "b12237",
 "base_model": "Qwen/Qwen2.5-VL-3B-Instruct@66285546d2b821cf421d4f5eb2576359d3770cd3",
 "adapter": "AbrarAlam/disasterm3-qwen25vl3b-l2@13ce20ba6d0a13dc5a5ddf2ad09e0704897e4d5c (c/final)",
 "llama_cpp_zip_sha256": "1472bc5767155a77811395c7a5504283e3eb7c0aedff113e42d485d3ae00f8c7"
}


In [2]:
# ── Cell 2: The benchmark's prompts and scorer, copied unchanged from l3-4bit-3b-handoff.ipynb (Cell 2) ──
PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})
# ── The benchmark's prompts and scorer, and the 3Bs (the same code in steps 3 and 4) ──
# The four prompts, copied from L0 step 2 (which copied them from the DisasterM3 authors' runner)
PROMPTS = dict(
    bearing_body="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option(s) from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    landuse="""Analyze the image to answer the following question. Choose the best option(s) from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letters of the correct options, separated by a comma and a space (e.g., C, D, H). Do not include any explanation or other text.""",
    single_choice="""Analyze both the pre-disaster and post-disaster images to answer the following question. Choose the best option from the candidate options provided.

pre-disaster image:
<image>

post-disaster image:
<image>

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
    relational_reasoning_qa="""Analyze the image to answer the following question. Choose the best option from the candidate options provided.

Question: {prompt}
Options: {options_str}

Your task is to respond with ONLY the capital letter of the correct option (e.g., C). Do not include any explanation or other text.""",
)
# (code, subset name in the 499A scorer, task name in the benchmark, task name in train_release.json)
TASKS = [("DTR", "disaster_type", "Disaster Type Recognition", "Disaster Type Recognition"),
         ("BBR", "bearing_body", "Disaster Bearing Bodies Recognition", "Disaster Bearing Bodies Recognition"),
         ("BDC", "building_damage_counting", "Building Damage Counting", "Building Damage Counting"),
         ("DRE", "road_damage_counting", "Road Damage Counting", "Road Damage Counting"),
         ("DSR", "landuse", "Disaster Scene Recognition", "Disaster Scene Recognition"),
         ("ORR", "relational_reasoning_qa", "Relational Reasoning", "relational reasoning")]
COUNTING = ("BDC", "DRE")

def letters(text):
    """The 499A scorer: the set of capital letters left after removing 'THE ANSWER IS' and 'OPTION'."""
    t = str(text).upper().replace("THE ANSWER IS", "").replace("OPTION", "")
    return set(re.findall(r"[A-Z]", t))

def prompt_text(e, subset):
    """(the benchmark prompt, the image keys in order), as L0 step 2's build() makes them."""
    p = e.get("prompts", "")
    p = p[0] if isinstance(p, list) else p
    if subset == "landuse":
        return PROMPTS["landuse"].format(prompt=p, options_str=e["options_str"]), ["pre_image_path"]
    if subset == "relational_reasoning_qa":           # its entries spell the options' field option_str
        return PROMPTS[subset].format(prompt=p, options_str=e["option_str"]), ["image_path"]
    key = "bearing_body" if subset == "bearing_body" else "single_choice"
    return PROMPTS[key].format(prompt=p, options_str=e["options_str"]), ["pre_image_path", "post_image_path"]

def content_of(text, n_images):
    """L0 step 2's layout: single-image prompts put the image first; two-image prompts fill each '<image>' gap."""
    parts = text.split("<image>")
    if len(parts) == 1:
        return [{"type": "image"} for _ in range(n_images)] + [{"type": "text", "text": text}]
    content = []
    for n, part in enumerate(parts):
        if part:
            content.append({"type": "text", "text": part})
        if n < len(parts) - 1:
            content.append({"type": "image"})
    return content

def option_values(e):
    """The five option values of a counting question, as numbers ('5.22%' -> 5.22)."""
    o = e["options_list"]
    o = ast.literal_eval(o) if isinstance(o, str) else o
    return [float(str(x).rstrip("%")) for x in o]

def no_image_rule(values):
    """Never sees the image: largest option above 20 -> the middle value, otherwise the second smallest (benchmark check, 7 Nov)."""
    s = sorted(values)
    return "ABCDE"[values.index(s[2] if s[4] > 20 else s[1])]

def chosen_value(answer, values):
    """The value of the one option an answer names; None if it names no option or several."""
    got = letters(answer)
    if len(got) != 1 or next(iter(got)) not in "ABCDE"[:len(values)]:
        return None
    return values["ABCDE".index(next(iter(got)))]

assert letters("The answer is D.") == {"D"} and no_image_rule([231, 323, 277, 139, 185]) == "A"
assert chosen_value("B", [1, 2, 3, 4, 5]) == 2 and chosen_value("B, C", [1, 2, 3, 4, 5]) is None
assert content_of("x<image>y<image>z", 2)[1] == {"type": "image"} and content_of("q", 1)[0] == {"type": "image"}


In [3]:
# ── Cell 3: Prepare the model once: merge the adapter, convert to llama.cpp's format, 4 bits ──
MERGE = r'''
import shutil, sys
from pathlib import Path
base, adapter, out = map(Path, sys.argv[1:4])
weights = out / "model.safetensors"
if weights.exists() or (out / "model.safetensors.index.json").exists():
    from safetensors import safe_open
    n = 0
    for f in ([weights] if weights.exists() else sorted(out.glob("model-*.safetensors"))):
        with safe_open(str(f), "pt") as s:
            n += len(list(s.keys()))
    print(f"merged weights already saved: {n} tensors", flush=True)
else:
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration
    from peft import PeftModel
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(base, dtype=torch.bfloat16, low_cpu_mem_usage=True)
    model = PeftModel.from_pretrained(model, adapter).merge_and_unload()
    model.save_pretrained(out, safe_serialization=True)
    print("merged into", out, flush=True)
# The processor's files (tokenizer, image settings, chat template) come unchanged from the base model.
# AutoProcessor would need torchvision for Qwen's video part, which this PC's Python doesn't have.
for f in base.iterdir():
    if f.is_file() and f.suffix in (".json", ".txt", ".jinja") and "safetensors" not in f.name and not (out / f.name).exists():
        shutil.copy2(f, out / f.name)
        print("copied", f.name, flush=True)
'''

def step(name, args, done):
    if done.exists():
        print(f"✓ {name}: already there ({done.name})")
        return
    t = time.time()
    log = OUT / f"prepare_{name}.log"
    with open(log, "w", encoding="utf-8") as f:
        r = subprocess.run([str(a) for a in args], stdout=f, stderr=subprocess.STDOUT, text=True)
    assert r.returncode == 0 and done.exists(), f"{name} failed; see {log}"
    print(f"✓ {name}: {(time.time() - t) / 60:.1f} min")

GGUF.mkdir(parents=True, exist_ok=True)
step("merge", [sys.executable, "-c", MERGE, BASE, ADAPTER, MERGED], MERGED / "preprocessor_config.json")
step("convert_language", [sys.executable, CONVERT, MERGED, "--outtype", "f16", "--outfile", LLM_F16], LLM_F16)
step("convert_image", [sys.executable, CONVERT, MERGED, "--mmproj", "--outtype", "f16", "--outfile", MMPROJ], MMPROJ)
step("quantize", [LLAMA / "llama-quantize.exe", LLM_F16, LLM_Q4, "Q4_K_M"], LLM_Q4)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""):
            h.update(b)
    return h.hexdigest()

FILES = {p.name: {"gb": round(p.stat().st_size / 1e9, 3), "sha256": sha256(p)} for p in (LLM_Q4, MMPROJ)}
DISK_GB = round(sum(v["gb"] for v in FILES.values()), 2)
print(json.dumps(FILES, indent=1), f"\nmodel on disk: {DISK_GB} GB")


✓ merge: 0.4 min


✓ convert_language: 2.3 min


✓ convert_image: 1.9 min


✓ quantize: 5.2 min


{
 "3b-c-Q4_K_M.gguf": {
  "gb": 1.93,
  "sha256": "986e3ded28b312d2e954af3fe9661e860501759898214faeb55a018fabc84cd3"
 },
 "mmproj-3b-c-f16.gguf": {
  "gb": 1.338,
  "sha256": "dd9219a4401c596ca9a5f67a108e218847801ea250b1ec2dc7932facda8adeda"
 }
} 
model on disk: 3.27 GB


In [4]:
# ── Cell 4: The 360 questions, built as L3 built them, with the photos on this PC ──
sel = json.loads((INPUTS / "questions.json").read_text(encoding="utf-8"))
bench = json.loads((INPUTS / "benchmark_release.json").read_text(encoding="utf-8"))

def sensor(e):
    return e.get("post_image_type") or e.get("image_type") or "Optical"

rows = []
for code, subset, bench_task, _ in TASKS:
    entries = [(f"{subset}_{i}", e) for i, e in enumerate(e for e in bench if e.get("task") == bench_task)]
    for qid, e in entries:
        if sensor(e) != "Optical":
            continue
        text, keys = prompt_text(e, subset)
        rows.append({"id": qid, "task": code, "text": text, "names": [name_of(e[k]) for k in keys],
                     "truth": e["ground_truth_option"]})
Q = pd.DataFrame(rows).set_index("id")
T4 = pd.read_csv(L3_DIR / "per_question.csv").set_index("id")          # the 4-bit 3B on a T4 (L3)
assert len(Q) == len(T4) == 10675, (len(Q), len(T4))
SPEED, ACC = sel["speed"], sel["accuracy"]
need = {n for q in SPEED + ACC for n in Q.loc[q, "names"]}
for _ in range(120):                         # up to an hour, if l3-step6-download-inputs.py is still running
    local = {p.name: p for p in (INPUTS / "test_images").rglob("*.png")}
    if need <= set(local):
        break
    print(f"  waiting for photos: {len(need - set(local))} still missing", flush=True)
    time.sleep(30)
for qid in SPEED + ACC:                                                 # same question, same photos, same truth
    assert [name_of(p) for p in ast.literal_eval(T4.loc[qid, "images"])] == Q.loc[qid, "names"], qid
    assert T4.loc[qid, "truth"] == Q.loc[qid, "truth"], qid
    assert all(n in local for n in Q.loc[qid, "names"]), f"{qid}: a photo is missing; run l3-step6-download-inputs.py"
speed_l3 = json.loads((L3_DIR / "speed.json").read_text(encoding="utf-8"))["3b_nf4"]
assert len(speed_l3["answers"]) == len(SPEED) == 60
print(f"✓ {len(SPEED)} speed and {len(ACC)} accuracy questions, {Q.loc[SPEED + ACC, 'task'].value_counts().to_dict()}; "
      f"all {len({n for q in SPEED + ACC for n in Q.loc[q, 'names']})} photos on this PC")


  waiting for photos: 109 still missing


  waiting for photos: 104 still missing


  waiting for photos: 99 still missing


  waiting for photos: 93 still missing


  waiting for photos: 88 still missing


  waiting for photos: 83 still missing


... (12 outputs left out)

  waiting for photos: 15 still missing


  waiting for photos: 10 still missing


  waiting for photos: 4 still missing


✓ 60 speed and 300 accuracy questions, {'DTR': 60, 'BBR': 60, 'BDC': 60, 'DRE': 60, 'DSR': 60, 'ORR': 60}; all 519 photos on this PC


In [5]:
# ── Cell 5: Start the llama.cpp server, and sample its memory every 0.25 s ──
SERVER_LOG = OUT / "server.log"
cmd = [str(LLAMA / "llama-server.exe"), "-m", str(LLM_Q4), "--mmproj", str(MMPROJ), "--host", "127.0.0.1",
       "--port", str(PORT), "-c", str(CONTEXT), "-np", "1", "--jinja",
       "--image-max-tokens", str(MAX_PIXELS // (28 * 28))]   # the T4's cap: 768 image tokens (photos are also pre-scaled by sized())
t0 = time.time()
server = subprocess.Popen(cmd, stdout=open(SERVER_LOG, "w", encoding="utf-8"), stderr=subprocess.STDOUT)
URL = f"http://127.0.0.1:{PORT}"
while True:
    assert server.poll() is None, f"the server stopped; see {SERVER_LOG}"
    try:
        if requests.get(f"{URL}/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    assert time.time() - t0 < 600, "the server did not start in 10 minutes"
    time.sleep(1)
LOAD_S = round(time.time() - t0, 1)

proc = psutil.Process(server.pid)
samples, stop = [], threading.Event()
def sample():
    while not stop.is_set():
        try:
            samples.append((time.time(), proc.memory_info().rss))
        except psutil.Error:
            break
        stop.wait(0.25)
threading.Thread(target=sample, daemon=True).start()
log_text = SERVER_LOG.read_text(encoding="utf-8", errors="replace")
N_THREADS = (re.findall(r"n_threads\s*=\s*(\d+)", log_text) or ["?"])[0]
print(f"✓ server ready in {LOAD_S} s, {N_THREADS} threads; memory now {proc.memory_info().rss / 1e9:.2f} GB")


✓ server ready in 14.8 s, 14 threads; memory now 4.19 GB


In [6]:
# ── Cell 6: Ask the 60 speed questions, then the 300 accuracy questions, one at a time (resumes) ──
ANSWERS = OUT / "answers.jsonl"
done = {}
if ANSWERS.exists():
    for line in ANSWERS.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        done[r["id"]] = r

def as_png(path):
    img = sized(Image.open(path).convert("RGB"))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return img.size, "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def ask(qid):
    q = Q.loc[qid]
    sizes, parts, images = [], [], [as_png(local[n]) for n in q["names"]]
    for item in content_of(q["text"], len(images)):            # L0 step 2's layout, as on the T4
        if item["type"] == "text":
            parts.append({"type": "text", "text": item["text"]})
        else:
            size, url = images[len(sizes)]
            sizes.append(size)
            parts.append({"type": "image_url", "image_url": {"url": url}})
    body = {"messages": [{"role": "user", "content": parts}], "temperature": 0, "top_k": 1, "seed": 0,
            "max_tokens": MAX_NEW_TOKENS, "cache_prompt": False}
    t = time.time()
    r = requests.post(f"{URL}/v1/chat/completions", json=body, timeout=1800)
    seconds = time.time() - t
    r.raise_for_status()
    out = r.json()
    tm = out.get("timings", {})
    return {"id": qid, "set": "speed" if qid in SPEED else "accuracy", "task": q["task"],
            "answer": out["choices"][0]["message"]["content"].strip(), "seconds": round(seconds, 3),
            "prompt_tokens": tm.get("prompt_n"), "prompt_ms": tm.get("prompt_ms"),
            "answer_tokens": tm.get("predicted_n"), "answer_ms": tm.get("predicted_ms"),
            "image_sizes": sizes, "memory_gb_after": round(proc.memory_info().rss / 1e9, 3)}

todo = [q for q in SPEED + ACC if q not in done]
print(f"{len(done)} done, {len(todo)} to go")
with open(ANSWERS, "a", encoding="utf-8") as f:
    for n, qid in enumerate(todo, 1):
        r = ask(qid)
        done[qid] = r
        f.write(json.dumps(r) + "\n")
        f.flush()
        if n % 20 == 0 or n == len(todo):
            recent = [done[q]["seconds"] for q in todo[max(0, n - 20):n]]
            print(f"  {len(done)}/{len(SPEED) + len(ACC)}  last 20: {sum(recent) / len(recent):.1f} s per question", flush=True)
stop.set()
time.sleep(0.5)
mi = proc.memory_info()
PEAK = {"peak_working_set_gb": round(getattr(mi, "peak_wset", max(s for _, s in samples)) / 1e9, 2),
        "peak_sampled_gb": round(max(s for _, s in samples) / 1e9, 2),
        "peak_private_gb": round(getattr(mi, "peak_pagefile", 0) / 1e9, 2)}
server.terminate()
server.wait(timeout=60)
print("✓ all answered;", PEAK)


0 done, 360 to go


  20/360  last 20: 102.0 s per question


  40/360  last 20: 99.9 s per question


  60/360  last 20: 50.2 s per question


  80/360  last 20: 97.5 s per question


  100/360  last 20: 96.1 s per question


... (10 outputs left out)

  320/360  last 20: 74.7 s per question


  340/360  last 20: 52.5 s per question


  360/360  last 20: 52.7 s per question


✓ all answered; {'peak_working_set_gb': 4.67, 'peak_sampled_gb': 4.62, 'peak_private_gb': 12.15}


In [7]:
# ── Cell 7: Scores against the T4's 4-bit answers, speed and memory, expectation against result, files ──
A = pd.DataFrame([done[q] for q in SPEED + ACC]).set_index("id")
A["truth"] = Q.loc[A.index, "truth"]
A["correct"] = [letters(a) == letters(t) for a, t in zip(A["answer"], A["truth"])]
A["t4_answer"] = T4.loc[A.index, "answer"].fillna("")
A["t4_correct"] = T4.loc[A.index, "correct"].astype(bool)
A["same_answer"] = [letters(a) == letters(b) for a, b in zip(A["answer"], A["t4_answer"])]
A["garbled"] = [bool(re.fullmatch(r"!+", a)) or not a for a in A["answer"]]

def md(df, index=True):
    d = df.reset_index() if index else df
    head = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns)
    return head + "".join("\n| " + " | ".join(map(str, r)) + " |" for r in d.itertuples(index=False))

def rate(tokens, ms):
    t, s = pd.to_numeric(tokens, errors="coerce").sum(), pd.to_numeric(ms, errors="coerce").sum() / 1000
    return round(t / s, 1) if s > 0 else None

def mcnemar(only_a, only_b):
    n = only_a + only_b
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, k) for k in range(min(only_a, only_b) + 1)) / 2 ** n)

only_laptop = int((A["correct"] & ~A["t4_correct"]).sum())
only_t4 = int((~A["correct"] & A["t4_correct"]).sum())
P_VALUE = mcnemar(only_laptop, only_t4)
by_task = A.groupby("task").agg(questions=("correct", "size"), laptop_pct=("correct", "mean"), t4_pct=("t4_correct", "mean"),
                                same_answer_pct=("same_answer", "mean"), seconds_mean=("seconds", "mean"))
by_task[["laptop_pct", "t4_pct", "same_answer_pct"]] *= 100
by_task = by_task.reindex([t[0] for t in TASKS]).round(2)
sp = A.loc[SPEED]
speed = {"questions": len(sp), "s_per_question_mean": round(sp["seconds"].mean(), 2),
         "s_per_question_median": round(sp["seconds"].median(), 2),
         "t4_s_per_question_mean": 1.719,
         "prompt_tokens_mean": round(pd.to_numeric(sp["prompt_tokens"], errors="coerce").mean(), 1),
         "prompt_tokens_per_s": rate(sp["prompt_tokens"], sp["prompt_ms"]),
         "answer_tokens_per_s": rate(sp["answer_tokens"], sp["answer_ms"]),
         "same_answer_as_t4_speed_run_pct": round(100 * sum(letters(a) == letters(b) for a, b in zip(sp["answer"], speed_l3["answers"])) / len(sp), 1)}
s_all = round(A["seconds"].mean(), 2)
exp = pd.DataFrame([
    ("peak memory at most 6 GB (likely 3 to 5)", f"{PEAK['peak_working_set_gb']} GB", PEAK["peak_working_set_gb"] <= 6),
    ("10 to 60 s per question", f"{speed['s_per_question_mean']} s (speed set), {s_all} s (all 360)", 10 <= speed["s_per_question_mean"] <= 60),
    ("no difference beyond chance from the T4's 4-bit answers", f"laptop {A['correct'].mean() * 100:.2f}% against T4 {A['t4_correct'].mean() * 100:.2f}%; only laptop right {only_laptop}, only T4 {only_t4}, p = {P_VALUE:.3g}", P_VALUE >= 0.05),
    ("no garbled or empty answers", f"{int(A['garbled'].sum())}", int(A["garbled"].sum()) == 0),
], columns=["expectation (24 Nov)", "result", "met"])
summary = {"machine": MACHINE, "model_files": FILES, "disk_gb": DISK_GB, "load_s": LOAD_S, "threads": N_THREADS,
           "memory": PEAK, "speed": speed, "accuracy_pct": {"laptop": round(A["correct"].mean() * 100, 2), "t4": round(A["t4_correct"].mean() * 100, 2)},
           "mcnemar": {"only_laptop_right": only_laptop, "only_t4_right": only_t4, "p": P_VALUE},
           "same_answer_pct": round(A["same_answer"].mean() * 100, 1), "garbled": int(A["garbled"].sum()),
           "server_command": " ".join(Path(c).name if os.sep in c or "/" in c else c for c in cmd)}
A.drop(columns=["image_sizes"]).to_csv(OUT / "per_question.csv")
by_task.to_csv(OUT / "by_task.csv")
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=str), encoding="utf-8")
readme = f'''# L3 step 6: the 4-bit 3B on a laptop with no GPU

From `l3-step6-laptop-cpu.ipynb`, run in Jupyter on the PC ({time.strftime("%d %b %Y")}). The rule and the expectation are fixed in the notebook's first cell. Questions: `l3-step6-download-inputs.py`.

## Machine and program

{MACHINE["cpu"]}, {MACHINE["cores"]} cores, {MACHINE["memory_gb"]} GB of memory, no NVIDIA GPU; llama.cpp {MACHINE["llama_cpp"]} on the CPU, {N_THREADS} threads. Model files: {DISK_GB} GB on disk (the language part in 4 bits, Q4_K_M; the image part in 16 bits). Server ready in {LOAD_S} s.

## Memory and speed

| | Laptop CPU | T4 (L3) |
|---|---|---|
| Peak memory | {PEAK["peak_working_set_gb"]} GB (working set) | 3.03 GB (GPU) |
| Seconds per question, mean (median), the 60 L3 timed | {speed["s_per_question_mean"]} ({speed["s_per_question_median"]}) | 1.72 (1.91) |
| Prompt reading | {speed["prompt_tokens_per_s"]} tokens/s, {speed["prompt_tokens_mean"]} tokens per question | — |
| Answer writing | {speed["answer_tokens_per_s"]} tokens/s | — |

## Accuracy on the same 360 questions

{md(by_task)}

Overall: laptop {A["correct"].mean() * 100:.2f}%, T4 {A["t4_correct"].mean() * 100:.2f}%. Only the laptop right on {only_laptop}, only the T4 on {only_t4} (exact McNemar p = {P_VALUE:.3g}). Same answer as the T4 on {A["same_answer"].mean() * 100:.1f}% of questions.

## Expectation against result

{md(exp, index=False)}

## Files

`per_question.csv`, `by_task.csv`, `summary.json`, `answers.jsonl`, `server.log`, `prepare_*.log`
'''
(OUT / "README.md").write_text(readme, encoding="utf-8")
print(by_task.to_string(), "\n")
print(exp.to_string(index=False))


      questions  laptop_pct  t4_pct  same_answer_pct  seconds_mean
task                                                              
DTR          60       53.33   55.00            90.00         96.05
BBR          60       10.00   11.67            68.33         97.92
BDC          60       51.67   46.67            86.67         98.01
DRE          60       40.00   43.33            88.33         99.04
DSR          60       35.00   35.00            65.00         51.97
ORR          60       48.33   53.33            83.33         52.42 

                                   expectation (24 Nov)                                                                       result   met
               peak memory at most 6 GB (likely 3 to 5)                                                                      4.67 GB  True
                                10 to 60 s per question                                       84.04 s (speed set), 82.57 s (all 360) False
no difference beyond chance from the T4's 4-b